# 04 — Sentiment analysis

Score each ad's creative body on emotional tone (positive / negative / neutral) so we can layer sentiment alongside the topic labels from notebook 03. Together they let us answer questions like:

- Are climate-themed ads more often *fearful* (negative framing) or *aspirational* (positive)?
- Which advertisers run the most negative attack-style content?
- Does sentiment correlate with spend or impressions per ad?

Runs independently of 03 — both consume the v2 parquet and emit a new column.

## Input

v2 parquet (`/user/s3348393/main/preprocessing/v2/parquet`).

Filter to `ad_seq_no = 1` (one row per ad — the latest snapshot) and `creative_bodies` non-empty (no body, no sentiment). Don't pre-filter on `match_type` here — sentiment is interesting across *all* categories (candidate / party_org / government / residual), not just the residual.

## Output

New column `sentiment` (`positive` / `negative` / `neutral`) plus optionally `sentiment_score` (continuous, e.g. compound score from VADER). Joined back to the corpus and written as v3 parquet.

Schema after this notebook runs:

```
v2 columns... + sentiment (string) + sentiment_score (float, optional)
```

## Pipeline

Different preprocessing to notebook 03 — sentiment needs negation and word order preserved, so the stop-words list must keep tokens like `not`, `never`, `no`, `but` that 03 strips.

1. **Text cleaning** — same regex normalisation as 03 (lowercase, strip URLs/punctuation/non-alphabetic).
2. **Tokenisation** — `RegexTokenizer` with the same `\W+` pattern.
3. **Custom stop-words removal** — English defaults *minus* negation modifiers. Build the list explicitly.
4. **N-grams** — `NGram(n=2)` (and optionally `n=3`) to capture phrases like "not happy", "cost of living", "don't trust".
5. **Sentiment scoring** — see decision below.
6. **Write** — append `sentiment` (and optionally `sentiment_score`) to a v3 parquet.

## Key decision: where do the labels come from?

Spark MLlib's classifiers (Logistic Regression, Naive Bayes) are supervised — they need training data. Three options, in increasing order of effort and tailoring:

### Option A — Lexicon-based (VADER or similar)

Use a pretrained lexicon scorer (VADER via `nltk` or `vaderSentiment`) as a UDF over the ad body. Each ad gets a continuous compound score in `[-1, +1]`, bucketed into pos/neg/neutral by threshold.

- **Pros**: zero training data needed, fast to wire up, runs entirely as a pandas UDF.
- **Cons**: lexicon was tuned on social-media data, not political ads. Will miss sarcasm and ad-specific framing.
- **Time**: ~half a day to wire and validate.

### Option B — Hand-labelled subset + train classifier

Hand-label 500–2,000 ads on pos/neg/neutral, train a Spark Logistic Regression or Naive Bayes on n-gram features, apply to the full corpus.

- **Pros**: tailored to political-ad language, captures domain-specific framing.
- **Cons**: 500+ ads of manual labelling is real work. Inter-annotator drift if more than one person labels.
- **Time**: 1–2 days of labelling plus model training.

### Option C — Transfer from a labelled corpus

Train on an external labelled dataset (e.g. Twitter US Airline Sentiment, IMDB) and apply to ads.

- **Pros**: large training set available off-the-shelf.
- **Cons**: domain mismatch — political ads aren't tweets aren't movie reviews. Likely worse than even Option A.
- **Verdict**: not recommended for this corpus.

### Recommendation

**Start with Option A (VADER).** It's the cheapest path to a working `sentiment` column. If the assignment requires a *supervised classifier* specifically, do Option A first to establish a baseline and have a fallback, then layer Option B on top once topic labels from notebook 03 are settled — those topic labels make it much easier to stratify the hand-labelling sample.

## Open questions

- **Threshold for pos/neg/neutral** with VADER compound score — default cuts at ±0.05, but political ad language tends to be high-intensity, so ±0.2 might give a cleaner middle bucket. Decide after eyeballing a sample.
- **Multi-creative ads** — some ads have multiple `creative_bodies` entries. Score each and aggregate (mean / max-magnitude) or just use the first? Use the first for parity with notebook 03.
- **Validation** — sanity check by sampling 50 ads per sentiment bucket and reading them. Are negative ads actually negative? Are neutrals actually neutral?
- **Sentiment + topic interaction** — once 03 and 04 are both done, the most interesting analyses live in the *cross-tab* of `topic_label × sentiment`. That's notebook 05 territory.

## Status

**Plan only.** Implementation comes after notebook 03 (topic modelling) is settled — sentiment can layer on top once we know what's commercial-vs-political and what the topic clusters look like.